# 12.3 记忆 (Memory)

记忆系统使Agent能在交互过程中保持和利用历史信息，是实现长期一致性的关键。

本节涵盖：
- 短期记忆（上下文窗口管理）
- 长期记忆（向量存储与语义检索）
- 工作记忆（草稿板机制）
- 情景记忆与时间衰减
- 记忆整合与重要性评分
- MemGPT式分层记忆管理

## 1. 短期记忆 — 上下文窗口管理

**短期记忆**即当前对话的上下文窗口，存储最近的交互历史。

**核心挑战**：上下文长度有限，需要策略性管理：
- **滑动窗口**：保留最近K轮对话
- **摘要压缩**：对早期对话生成摘要，释放token空间
- **优先级排序**：按相关性保留最重要的消息

**产业实践**：
- Claude的滑动窗口+摘要策略
- GPT-4的128K上下文仍需管理策略
- MemGPT的虚拟上下文管理

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from collections import OrderedDict

torch.manual_seed(42)

class ShortTermMemory:
    def __init__(self, d_model=128, max_tokens=1024, summary_ratio=0.3):
        self.d_model = d_model
        self.max_tokens = max_tokens
        self.summary_ratio = summary_ratio
        self.messages = []
        self.token_counts = []
        self.summary = None
        self.summary_encoder = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model))

    def add_message(self, role, content_embed, token_count=10):
        self.messages.append({'role': role, 'content': content_embed, 'tokens': token_count})
        self.token_counts.append(token_count)
        if self.total_tokens() > self.max_tokens:
            self._compress()

    def total_tokens(self):
        return sum(self.token_counts)

    def _compress(self):
        n_to_summarize = max(1, len(self.messages) // 2)
        old_embeds = torch.cat([m['content'] for m in self.messages[:n_to_summarize]], dim=0)
        summary_embed = self.summary_encoder(old_embeds.mean(dim=0, keepdim=True))
        self.summary = summary_embed
        summary_tokens = int(sum(self.token_counts[:n_to_summarize]) * self.summary_ratio)
        self.messages = self.messages[n_to_summarize:]
        self.token_counts = self.token_counts[n_to_summarize:]
        self.messages.insert(0, {'role': 'system', 'content': summary_embed, 'tokens': summary_tokens})
        self.token_counts.insert(0, summary_tokens)

    def get_context(self, max_tokens=None):
        limit = max_tokens or self.max_tokens
        context = []
        used = 0
        if self.summary is not None:
            context.append(self.summary)
            used += self.token_counts[0] if self.token_counts else 0
        for msg in reversed(self.messages):
            if used + msg['tokens'] > limit:
                break
            context.append(msg['content'])
            used += msg['tokens']
        context.reverse()
        return context

stm = ShortTermMemory(d_model=128, max_tokens=100)

print('=== Short-Term Memory (Context Window Management) ===')
for i in range(15):
    msg_embed = torch.randn(1, 128) + i * 0.1
    stm.add_message('user' if i % 2 == 0 else 'assistant', msg_embed, token_count=15)
    print(f'Added message {i+1}: total_tokens={stm.total_tokens()}, messages={len(stm.messages)}')

context = stm.get_context()
print(f'\nFinal context: {len(context)} messages, {stm.total_tokens()} tokens')
print(f'Summary exists: {stm.summary is not None}')
print(f'\nKey: When context exceeds max_tokens, older messages are compressed into a summary.')
print(f'This preserves information while fitting within the context window.')

## 2. 长期记忆 — 向量存储与语义检索

**长期记忆**跨会话持久化存储信息，通过语义相似度检索相关记忆。

**核心组件**：
1. **编码器**：将记忆内容编码为向量
2. **索引**：高效的向量相似度搜索（FAISS/HNSW）
3. **检索**：根据查询向量找到最相关的K条记忆
4. **写入**：新记忆的存储与去重

**产业实践**：
- ChromaDB、Pinecone、Weaviate等向量数据库
- FAISS用于十亿级向量检索
- HNSW用于实时近似最近邻搜索

In [ ]:
class VectorMemory(nn.Module):
    def __init__(self, d_model=128, d_key=64, capacity=500, decay_rate=0.995):
        super().__init__()
        self.d_model = d_model
        self.d_key = d_key
        self.capacity = capacity
        self.decay_rate = decay_rate
        self.key_encoder = nn.Sequential(nn.Linear(d_model, d_key), nn.LayerNorm(d_key))
        self.value_encoder = nn.Sequential(nn.Linear(d_model, d_key), nn.LayerNorm(d_key))
        self.register_buffer('keys', torch.zeros(capacity, d_key))
        self.register_buffer('values', torch.zeros(capacity, d_key))
        self.register_buffer('timestamps', torch.zeros(capacity))
        self.register_buffer('importance', torch.zeros(capacity))
        self.register_buffer('access_count', torch.zeros(capacity))
        self.size = 0
        self.current_time = 0

    def store(self, key, value, importance_score=0.5):
        if self.size < self.capacity:
            idx = self.size
            self.size += 1
        else:
            scores = self.importance[:self.size] * self.decay_rate ** (self.current_time - self.timestamps[:self.size])
            idx = scores.argmin().item()

        self.keys[idx] = F.normalize(self.key_encoder(key), dim=-1).detach()
        self.values[idx] = F.normalize(self.value_encoder(value), dim=-1).detach()
        self.timestamps[idx] = self.current_time
        self.importance[idx] = importance_score
        self.access_count[idx] = 0
        self.current_time += 1

    def retrieve(self, query, top_k=5, recency_weight=0.1, importance_weight=0.2):
        if self.size == 0:
            return torch.zeros(0, self.d_key), torch.zeros(0)
        q = F.normalize(self.key_encoder(query), dim=-1)
        n_stored = min(self.size, self.capacity)
        similarity = q @ self.keys[:n_stored].T
        recency = self.decay_rate ** (self.current_time - self.timestamps[:n_stored])
        combined = similarity + recency_weight * recency + importance_weight * self.importance[:n_stored]
        top_k = min(top_k, n_stored)
        top_indices = combined.topk(top_k).indices
        top_scores = combined.gather(0, top_indices)
        self.access_count[top_indices] += 1
        return self.values[top_indices], top_scores

    def consolidate(self, threshold=0.95):
        if self.size < 2:
            return 0
        n = min(self.size, self.capacity)
        sim_matrix = self.keys[:n] @ self.keys[:n].T
        merged = 0
        to_remove = set()
        for i in range(n):
            if i in to_remove:
                continue
            for j in range(i + 1, n):
                if j in to_remove:
                    continue
                if sim_matrix[i, j] > threshold:
                    self.values[i] = (self.values[i] + self.values[j]) / 2
                    self.importance[i] = max(self.importance[i].item(), self.importance[j].item())
                    to_remove.add(j)
                    merged += 1
        return merged

memory = VectorMemory(d_model=128, d_key=64, capacity=100, decay_rate=0.995)

print('=== Long-Term Memory (Vector Store with Decay & Importance) ===')
for i in range(20):
    key = torch.randn(1, 128)
    value = torch.randn(1, 128) + i * 0.05
    importance = 0.3 + 0.7 * (i / 20)
    memory.store(key, value, importance_score=importance)

query = torch.randn(1, 128)
retrieved, scores = memory.retrieve(query, top_k=5)

print(f'Stored memories: {memory.size}')
print(f'Top-5 retrieval scores: {[f"{s:.3f}" for s in scores.tolist()]}')
print(f'Access counts: {memory.access_count[:20].int().tolist()}')

merged = memory.consolidate(threshold=0.9)
print(f'\nConsolidation: merged {merged} similar memories')
print(f'\nKey: Memory retrieval combines semantic similarity, recency decay, and importance.')
print(f'Consolidation merges near-duplicate memories to save capacity.')

## 3. 工作记忆 — 草稿板机制

**工作记忆**是Agent在推理过程中的临时读写空间，类似人类的"草稿纸"。

**与短期/长期记忆的区别**：
- 短期记忆：对话历史（只读，自动维护）
- 长期记忆：持久化知识（检索式，需要显式存储）
- 工作记忆：当前任务的临时空间（读写，手动管理）

**典型用途**：
- 存储中间计算结果
- 维护推理步骤的状态
- 缓存工具调用结果供后续步骤使用
- Scratchpad/Chain of Thought的显式化

In [ ]:
class WorkingMemory:
    def __init__(self, d_model=128, max_slots=16, eviction_strategy='lru'):
        self.d_model = d_model
        self.max_slots = max_slots
        self.eviction_strategy = eviction_strategy
        self.slots = OrderedDict()
        self.access_times = {}
        self.current_time = 0
        self.attention_gate = nn.Sequential(
            nn.Linear(d_model * 2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def write(self, key, value_tensor, metadata=None):
        if len(self.slots) >= self.max_slots and key not in self.slots:
            self._evict()
        self.slots[key] = {
            'value': value_tensor.detach().clone(),
            'metadata': metadata or {},
            'write_time': self.current_time
        }
        self.access_times[key] = self.current_time
        self.current_time += 1

    def read(self, key):
        if key in self.slots:
            self.access_times[key] = self.current_time
            self.current_time += 1
            return self.slots[key]['value'], self.slots[key]['metadata']
        return torch.zeros(1, self.d_model), {}

    def read_with_attention(self, query, top_k=3):
        if not self.slots:
            return torch.zeros(0, self.d_model), torch.zeros(0)
        keys = list(self.slots.keys())
        values = torch.cat([self.slots[k]['value'] for k in keys], dim=0)
        query_expanded = query.expand(values.size(0), -1)
        pairs = torch.cat([query_expanded, values], dim=-1)
        gates = self.attention_gate(pairs).squeeze(-1)
        scores = F.softmax(gates, dim=0)
        top_k = min(top_k, len(keys))
        top_indices = scores.topk(top_k).indices
        return values[top_indices], scores[top_indices]

    def _evict(self):
        if self.eviction_strategy == 'lru':
            oldest = min(self.access_times, key=self.access_times.get)
        elif self.eviction_strategy == 'fifo':
            oldest = next(iter(self.slots))
        else:
            oldest = min(self.access_times, key=self.access_times.get)
        del self.slots[oldest]
        del self.access_times[oldest]

    def list_keys(self):
        return list(self.slots.keys())

    def clear(self):
        self.slots.clear()
        self.access_times.clear()

wm = WorkingMemory(d_model=128, max_slots=5, eviction_strategy='lru')

print('=== Working Memory (Scratchpad with Attention Read) ===')
for i in range(8):
    key = f'step_{i}_result'
    value = torch.randn(1, 128) * (i + 1)
    wm.write(key, value, metadata={'step': i, 'type': 'computation'})
    print(f'Write: {key} (slots: {len(wm.slots)}/{wm.max_slots})')

print(f'\nCurrent keys: {wm.list_keys()}')
val, meta = wm.read('step_5_result')
print(f'Read step_5_result: norm={val.norm():.3f}, metadata={meta}')

query = torch.randn(1, 128)
values, scores = wm.read_with_attention(query, top_k=3)
print(f'\nAttention-based read (top-3):')
print(f'  Scores: {[f"{s:.3f}" for s in scores.tolist()]}')
print(f'  Values norm: {[f"{v.norm():.3f}" for v in values]}')

print(f'\nKey: Working memory supports both direct key-based and attention-based retrieval.')
print(f'LRU eviction ensures the most recently used items are preserved.')

## 4. 情景记忆与时间衰减

**情景记忆**记录具体的事件和经历，带有时间戳和上下文信息。

**时间衰减模型**：
- **指数衰减**：$w(t) = \lambda^t$，近期记忆权重高
- **幂律衰减**：$w(t) = (1+t)^{-\alpha}$，更符合人类遗忘曲线
- **分段衰减**：不同时间段使用不同衰减率

**重要性评分**：
- 情感强度：带情感色彩的事件更重要
- 频率：频繁出现的信息更重要
- 关联度：与当前任务相关的信息更重要
- 新颖度：首次出现的信息更重要

**产业参考**：Generative Agents (Stanford) 使用重要性+时效性+相关性三维评分。

In [ ]:
class EpisodicMemory(nn.Module):
    def __init__(self, d_model=128, capacity=200, decay_lambda=0.99):
        super().__init__()
        self.d_model = d_model
        self.capacity = capacity
        self.decay_lambda = decay_lambda
        self.content_encoder = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 32))
        self.importance_net = nn.Sequential(
            nn.Linear(d_model + 4, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.episodes = []
        self.current_time = 0

    def store_episode(self, content_embed, emotion=0.5, frequency=1, novelty=0.5, relevance=0.5):
        features = torch.cat([content_embed,
                              torch.tensor([[emotion, frequency, novelty, relevance]])], dim=-1)
        importance = self.importance_net(features).item()
        episode = {
            'content': content_embed.detach().clone(),
            'encoded': self.content_encoder(content_embed).detach().clone(),
            'timestamp': self.current_time,
            'importance': importance,
            'emotion': emotion,
            'frequency': frequency,
            'access_count': 0
        }
        self.episodes.append(episode)
        if len(self.episodes) > self.capacity:
            self._evict_episode()
        self.current_time += 1
        return importance

    def retrieve(self, query_embed, top_k=5, recency_weight=0.3, importance_weight=0.3, relevance_weight=0.4):
        if not self.episodes:
            return [], []
        query_encoded = self.content_encoder(query_embed)
        scores = []
        for ep in self.episodes:
            relevance = F.cosine_similarity(query_encoded, ep['encoded'], dim=-1).item()
            time_diff = self.current_time - ep['timestamp']
            recency = self.decay_lambda ** time_diff
            importance = ep['importance']
            score = (relevance_weight * relevance +
                     recency_weight * recency +
                     importance_weight * importance)
            scores.append(score)
        scores = torch.tensor(scores)
        top_k = min(top_k, len(self.episodes))
        top_indices = scores.topk(top_k).indices
        for idx in top_indices:
            self.episodes[idx]['access_count'] += 1
        return [self.episodes[i] for i in top_indices], [scores[i].item() for i in top_indices]

    def _evict_episode(self):
        scores = []
        for ep in self.episodes:
            time_diff = self.current_time - ep['timestamp']
            recency = self.decay_lambda ** time_diff
            score = ep['importance'] * recency + 0.1 * ep['access_count']
            scores.append(score)
        worst_idx = torch.tensor(scores).argmin().item()
        self.episodes.pop(worst_idx)

    def reflect(self, query_embed, n_episodes=10):
        episodes, scores = self.retrieve(query_embed, top_k=n_episodes)
        if not episodes:
            return None
        contents = torch.cat([ep['content'] for ep in episodes], dim=0)
        weights = torch.tensor(scores).unsqueeze(1)
        weights = weights / weights.sum()
        reflection = (contents * weights).sum(dim=0, keepdim=True)
        return reflection

episodic = EpisodicMemory(d_model=128, capacity=50, decay_lambda=0.99)

print('=== Episodic Memory with Time Decay ===')
events = [
    ('Morning meeting', 0.3, 1, 0.8, 0.9),
    ('Bug fix deployed', 0.7, 3, 0.3, 0.8),
    ('Team lunch', 0.8, 1, 0.5, 0.2),
    ('Code review', 0.2, 5, 0.1, 0.7),
    ('Production incident', 0.9, 1, 0.9, 0.95),
]
for name, emotion, freq, novelty, relevance in events:
    embed = torch.randn(1, 128)
    imp = episodic.store_episode(embed, emotion, freq, novelty, relevance)
    print(f'Stored: {name} -> importance={imp:.3f}')

query = torch.randn(1, 128)
results, scores = episodic.retrieve(query, top_k=3)
print(f'\nTop-3 retrieval:')
for i, (ep, score) in enumerate(zip(results, scores)):
    print(f'  {i+1}: score={score:.3f}, importance={ep["importance"]:.3f}, age={episodic.current_time - ep["timestamp"]}')

reflection = episodic.reflect(query)
print(f'\nReflection vector norm: {reflection.norm():.3f}')
print(f'\nKey: Episodic memory combines recency, importance, and relevance for retrieval.')
print(f'The reflect() method synthesizes past experiences into a single insight vector.')

## 5. MemGPT — 分层记忆管理

**MemGPT** (Memory-GPT) 借鉴操作系统的虚拟内存管理：
- **主记忆 (Main Context)**：当前活跃的上下文窗口（类似RAM）
- **归档记忆 (Archival Memory)**：持久化的长期存储（类似硬盘）
- **召回记忆 (Recall Memory)**：对话历史检索（类似日志）

**核心操作**：
- `memory.insert`：写入归档记忆
- `memory.search`：从归档记忆检索
- `core_memory_append`：追加到主记忆
- `core_memory_replace`：替换主记忆中的内容
- `archival_memory_insert`：插入归档记忆
- `archival_memory_search`：搜索归档记忆

**产业意义**：突破上下文窗口限制，实现"无限"记忆容量。

In [ ]:
class MemGPTSystem(nn.Module):
    def __init__(self, d_model=128, main_capacity=10, archival_capacity=500, recall_capacity=1000):
        super().__init__()
        self.d_model = d_model
        self.main_capacity = main_capacity
        self.archival_capacity = archival_capacity
        self.recall_capacity = recall_capacity
        self.key_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.LayerNorm(32))
        self.main_memory = OrderedDict()
        self.archival_keys = []
        self.archival_values = []
        self.recall_log = []
        self.page_faults = 0
        self.cache_hits = 0

    def core_memory_append(self, key, value_embed):
        if len(self.main_memory) >= self.main_capacity:
            self._evict_to_archival()
        self.main_memory[key] = value_embed.detach().clone()

    def core_memory_replace(self, key, value_embed):
        if key in self.main_memory:
            self.main_memory[key] = value_embed.detach().clone()
        else:
            self.core_memory_append(key, value_embed)

    def archival_memory_insert(self, content_embed, metadata=None):
        key = self.key_encoder(content_embed).detach()
        self.archival_keys.append(key)
        self.archival_values.append({
            'content': content_embed.detach().clone(),
            'metadata': metadata or {},
            'timestamp': len(self.archival_values)
        })
        if len(self.archival_keys) > self.archival_capacity:
            self.archival_keys.pop(0)
            self.archival_values.pop(0)

    def archival_memory_search(self, query_embed, top_k=5):
        if not self.archival_keys:
            return [], []
        q = F.normalize(self.key_encoder(query_embed), dim=-1)
        keys = torch.cat(self.archival_keys, dim=0)
        keys = F.normalize(keys, dim=-1)
        scores = q @ keys.T
        top_k = min(top_k, len(self.archival_keys))
        top_indices = scores.topk(top_k).indices
        return [self.archival_values[i] for i in top_indices], [scores[i].item() for i in top_indices]

    def _evict_to_archival(self):
        oldest_key = next(iter(self.main_memory))
        oldest_value = self.main_memory.pop(oldest_key)
        self.archival_memory_insert(oldest_value, metadata={'evicted_from': 'main', 'key': oldest_key})
        self.page_faults += 1

    def recall_search(self, query_embed, top_k=5):
        if not self.recall_log:
            return [], []
        q = self.key_encoder(query_embed)
        keys = torch.cat([self.key_encoder(r['content']) for r in self.recall_log], dim=0)
        keys = F.normalize(keys, dim=-1)
        q = F.normalize(q, dim=-1)
        scores = q @ keys.T
        top_k = min(top_k, len(self.recall_log))
        top_indices = scores.topk(top_k).indices
        return [self.recall_log[i] for i in top_indices], [scores[i].item() for i in top_indices]

    def add_to_recall(self, role, content_embed):
        self.recall_log.append({'role': role, 'content': content_embed.detach().clone()})
        if len(self.recall_log) > self.recall_capacity:
            self.recall_log.pop(0)

    def get_stats(self):
        return {
            'main_usage': f'{len(self.main_memory)}/{self.main_capacity}',
            'archival_usage': f'{len(self.archival_keys)}/{self.archival_capacity}',
            'recall_usage': f'{len(self.recall_log)}/{self.recall_capacity}',
            'page_faults': self.page_faults,
            'cache_hits': self.cache_hits
        }

memgpt = MemGPTSystem(d_model=128, main_capacity=5, archival_capacity=50)

print('=== MemGPT: Hierarchical Memory Management ===')
for i in range(8):
    key = f'fact_{i}'
    value = torch.randn(1, 128) + i * 0.1
    memgpt.core_memory_append(key, value)
    memgpt.add_to_recall('system', value)
    print(f'Added {key}: main={len(memgpt.main_memory)}/{memgpt.main_capacity}, '
          f'archival={len(memgpt.archival_keys)}')

print(f'\nMain memory keys: {list(memgpt.main_memory.keys())}')

query = torch.randn(1, 128)
arch_results, arch_scores = memgpt.archival_memory_search(query, top_k=3)
print(f'\nArchival search (top-3):')
for r, s in zip(arch_results, arch_scores):
    print(f'  score={s:.3f}, metadata={r["metadata"]}')

recall_results, recall_scores = memgpt.recall_search(query, top_k=3)
print(f'\nRecall search (top-3):')
for r, s in zip(recall_results, recall_scores):
    print(f'  role={r["role"]}, score={s:.3f}')

print(f'\nSystem stats: {memgpt.get_stats()}')
print(f'\nKey: MemGPT manages memory like an OS — main memory is fast but small,')
print(f'archival is large but requires search. Page faults occur when evicting from main.')

## 6. 记忆系统总结

| 记忆类型 | 容量 | 持久性 | 访问方式 | 典型实现 |
|---------|------|--------|---------|----------|
| 短期记忆 | 有限(上下文窗口) | 会话内 | 顺序读取 | 滑动窗口+摘要 |
| 长期记忆 | 大(向量DB) | 跨会话 | 语义检索 | FAISS/ChromaDB |
| 工作记忆 | 小(草稿板) | 任务内 | 键值/注意力 | Scratchpad |
| 情景记忆 | 中 | 跨会话 | 时间+语义 | 时间衰减+重要性 |
| 归档记忆 | 大 | 永久 | 语义检索 | MemGPT archival |

**设计原则**：
1. **分层管理**：不同类型的信息存储在不同层级
2. **衰减与遗忘**：模拟人类遗忘，保留重要信息
3. **整合与压缩**：定期合并相似记忆，节省空间
4. **重要性评分**：多维评分决定记忆保留优先级
5. **检索增强**：结合语义相似度、时效性、重要性

**前沿方向**：
- **MemoryBank**：基于Ebbinghaus遗忘曲线的记忆管理
- **SCM** (Structured Context Memory)：结构化上下文记忆
- **LongMem**：解耦记忆模块实现无限上下文
- **InfLLM**：通过记忆检索扩展上下文窗口